# Bank Marketing Classification — Full Project Notebook

This notebook implements the main technical requirements of the Machine Learning summative project:

- A real, open classification dataset with more than 20,000 raw rows
- Data inspection, cleaning decisions, exploratory analysis, and feature engineering
- Multiple Scikit-learn classifiers
- Cross-validated hyperparameter tuning
- A TensorFlow/Keras neural network using the `tf.data` API
- Precision, recall, F1, ROC-AUC, confusion matrices, ROC curves, and learning curves
- Held-out test evaluation, error analysis, and saved model artifacts

**Dataset:** UCI Bank Marketing. The expected full file is `bank-full.csv` (45,211 rows, 17 columns), usually found inside `bank+marketing.zip`.

**Prediction question:** Can we predict whether a customer subscribed to a term deposit (`y`) using information available before a phone call?

**Important modeling choice:** `duration` is deliberately excluded. It records call duration and is only known after the call, so including it would leak information for a pre-call prediction task. You may run a separate, clearly labeled experiment with it only if you explain that different prediction scenario.

**Learning and academic integrity:** Run the notebook yourself, review the outputs, and write the report in your own words based on the results you obtain. This notebook does not invent results in advance.

## 1. Install dependencies

Run this cell first. In Google Colab, TensorFlow is often preinstalled, but this makes sure the main packages are available.

In [ ]:
%pip -q install pandas numpy scikit-learn matplotlib seaborn tensorflow joblib

## 2. Import libraries and set reproducibility options

A fixed random seed makes many steps repeatable. Results can still vary slightly across hardware and library versions.

In [ ]:
import os
import io
import time
import random
import zipfile
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sklearn
import tensorflow as tf

from sklearn.model_selection import train_test_split, GridSearchCV, learning_curve
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    classification_report, ConfusionMatrixDisplay, RocCurveDisplay,
    precision_recall_curve, average_precision_score
)

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

ARTIFACT_DIR = Path("artifacts")
FIGURE_DIR = Path("figures")
ARTIFACT_DIR.mkdir(exist_ok=True)
FIGURE_DIR.mkdir(exist_ok=True)

print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("TensorFlow:", tf.__version__)
print("Working directory:", Path.cwd())

## 3. Get the dataset

You can upload either `bank-full.csv` **or the original `bank+marketing.zip`**. This cell accepts either one and searches nested ZIP archives for `bank-full.csv`.

If you work outside Colab, place `bank-full.csv` in the notebook's working folder or use the same upload widget supported by your environment.

In [ ]:
uploaded_dataset_path = None

try:
    from google.colab import files
    uploaded = files.upload()
    if uploaded:
        uploaded_dataset_path = next(iter(uploaded.keys()))
        print("Uploaded:", list(uploaded.keys()))
except ImportError:
    print("Not running in Colab. Looking for a local CSV or ZIP.")

In [ ]:
def find_bank_full_csv():
    # First check common local paths.
    candidates = [
        Path("bank-full.csv"),
        Path("/content/bank-full.csv"),
        Path("bank/bank-full.csv"),
        Path("bank+marketing/bank-full.csv"),
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate.read_bytes(), str(candidate)

    # If a ZIP was uploaded, inspect it, including nested ZIP files.
    if uploaded_dataset_path and Path(uploaded_dataset_path).exists():
        p = Path(uploaded_dataset_path)
        if p.suffix.lower() == ".csv":
            return p.read_bytes(), str(p)
        if p.suffix.lower() == ".zip":
            with zipfile.ZipFile(p) as zf:
                for name in zf.namelist():
                    if name.lower().endswith("bank-full.csv"):
                        return zf.read(name), f"{p}!/{name}"
                    if name.lower().endswith(".zip"):
                        try:
                            nested_bytes = zf.read(name)
                            with zipfile.ZipFile(io.BytesIO(nested_bytes)) as nested:
                                for nested_name in nested.namelist():
                                    if nested_name.lower().endswith("bank-full.csv"):
                                        return nested.read(nested_name), f"{p}!/{name}!/{nested_name}"
                        except zipfile.BadZipFile:
                            pass

    # Search files in the current folder for a CSV or ZIP if needed.
    for p in Path(".").glob("*.zip"):
        try:
            with zipfile.ZipFile(p) as zf:
                for name in zf.namelist():
                    if name.lower().endswith("bank-full.csv"):
                        return zf.read(name), f"{p}!/{name}"
                    if name.lower().endswith(".zip"):
                        try:
                            with zipfile.ZipFile(io.BytesIO(zf.read(name))) as nested:
                                for nested_name in nested.namelist():
                                    if nested_name.lower().endswith("bank-full.csv"):
                                        return nested.read(nested_name), f"{p}!/{name}!/{nested_name}"
                        except zipfile.BadZipFile:
                            pass
        except zipfile.BadZipFile:
            continue

    return None, None

csv_bytes, dataset_source = find_bank_full_csv()
if csv_bytes is None:
    raise FileNotFoundError(
        "Could not locate bank-full.csv. Upload bank-full.csv or bank+marketing.zip, "
        "or place bank-full.csv beside this notebook."
    )

# The UCI file is semicolon-separated, not comma-separated.
df = pd.read_csv(io.BytesIO(csv_bytes), sep=";")
print("Dataset source:", dataset_source)
print("Dataset shape:", df.shape)
display(df.head())

## 4. Check that the dataset meets the assignment requirements

The raw dataset must have at least 20,000 rows and a categorical target with two or more classes. Check those requirements before preprocessing.

In [ ]:
assert len(df) >= 20_000, f"Dataset has only {len(df):,} rows; minimum is 20,000."
assert "y" in df.columns, "Expected target column 'y' was not found."
assert df["y"].nunique(dropna=True) >= 2, "Target must have at least two classes."

print(f"PASS: {len(df):,} raw rows (minimum 20,000).")
print(f"PASS: categorical target 'y' has {df['y'].nunique()} classes: {sorted(df['y'].dropna().unique())}")
print("Columns:", len(df.columns))

## 5. Data quality inspection

Inspect data types, missing values, duplicates, and category distributions. Do not automatically delete duplicate rows or outliers without documenting a reason. The original dataset uses `unknown` as a category for some fields; it is not the same thing as a pandas missing value.

In [ ]:
print("Shape:", df.shape)
print("\nData types:")
display(df.dtypes.rename("dtype").to_frame())
print("\nMissing values:")
display(df.isna().sum().rename("missing_count").to_frame())
print("\nExact duplicate rows:", int(df.duplicated().sum()))
print("\nTarget counts:")
display(df["y"].value_counts().rename("count").to_frame())
print("\nTarget percentage:")
display((df["y"].value_counts(normalize=True) * 100).round(2).rename("percent").to_frame())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(data=df, x="y", ax=axes[0])
axes[0].set_title("Target class counts")
axes[0].set_xlabel("Term deposit subscription")
axes[0].set_ylabel("Number of customers")

target_pct = df["y"].value_counts(normalize=True).mul(100)
axes[1].bar(target_pct.index.astype(str), target_pct.values)
axes[1].set_title("Target class percentages")
axes[1].set_ylabel("Percent")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "target_distribution.png", dpi=160)
plt.show()

display(df.describe(include="number").T)

In [ ]:
for col in ["job", "marital", "education", "contact", "poutcome"]:
    if col in df.columns:
        print(f"\nCategory counts — {col}")
        display(df[col].value_counts(dropna=False).rename("count").to_frame())

## 6. Define the prediction problem and perform feature engineering

Target labels are converted to integers: `no = 0`, `yes = 1`.

For this project's main use case, predict subscription before contacting a customer. Therefore:
- Drop `duration`, because it is only known after the call.
- Add a simple engineered feature, `previously_contacted`, based on whether `previous` is greater than zero.
- Keep the original `previous` feature too, so the model can use the number of prior contacts.

The feature engineering decision should be explained in the report. We keep the raw dataset untouched in `df`.

In [ ]:
model_df = df.copy()

y = model_df["y"].map({"no": 0, "yes": 1})
if y.isna().any():
    raise ValueError("Unexpected or missing target values. Inspect df['y'].")

X = model_df.drop(columns=["y"]).copy()

# Avoid target leakage for the pre-call prediction task.
X = X.drop(columns=["duration"], errors="ignore")

# Feature engineering: has this customer been contacted in an earlier campaign?
if "previous" in X.columns:
    X["previously_contacted"] = (pd.to_numeric(X["previous"], errors="coerce") > 0).astype(int)

print("Features:", X.shape[1])
print("Duration excluded:", "duration" not in X.columns)
print("Engineered feature included:", "previously_contacted" in X.columns)
display(X.head())

## 7. Create train, validation, and test sets

Use a stratified 70/15/15 split:
- **Training set:** fit models and preprocessing.
- **Validation set:** compare models and choose settings.
- **Test set:** final evaluation after decisions are made.

Stratification preserves the class balance in each split. Preprocessing is fitted only on the training data to prevent data leakage.

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp
)

split_summary = pd.DataFrame({
    "split": ["train", "validation", "test"],
    "rows": [len(X_train), len(X_val), len(X_test)],
    "positive_rate": [y_train.mean(), y_val.mean(), y_test.mean()]
})
display(split_summary)

## 8. Build preprocessing pipelines

Numerical features use median imputation and standardization. Categorical features use most-frequent imputation and one-hot encoding. `handle_unknown='ignore'` allows the model to process categories not seen during training.

The same preprocessing logic is used for the baseline models, tuned Random Forest, and neural network. For hyperparameter tuning, preprocessing is placed inside the Scikit-learn pipeline so each cross-validation fold learns transformations only from its own training fold.

In [ ]:
numeric_cols = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = X_train.select_dtypes(exclude=["number"]).columns.tolist()

def make_preprocessor():
    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])
    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])
    return ColumnTransformer([
        ("numeric", numeric_pipe, numeric_cols),
        ("categorical", categorical_pipe, categorical_cols)
    ])

preprocessor = make_preprocessor()
X_train_p = preprocessor.fit_transform(X_train)
X_val_p = preprocessor.transform(X_val)
X_test_p = preprocessor.transform(X_test)

print("Numeric columns:", numeric_cols)
print("Categorical columns:", categorical_cols)
print("Processed feature matrix:", X_train_p.shape)

## 9. Train and compare traditional Scikit-learn models

We compare:
- Logistic Regression
- Random Forest
- Histogram-based Gradient Boosting

For a fair and easy-to-reproduce baseline, models are trained on the same preprocessed training data. Class imbalance means accuracy alone is not enough. Pay particular attention to positive-class precision, recall, F1, ROC-AUC, and average precision (PR-AUC-style summary).

In [ ]:
baseline_models = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000, class_weight="balanced", random_state=SEED
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200, class_weight="balanced",
        random_state=SEED, n_jobs=-1
    ),
    "HistGradientBoosting": HistGradientBoostingClassifier(
        max_iter=150, learning_rate=0.1, random_state=SEED
    )
}

def calculate_metrics(name, y_true, y_pred, y_prob):
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, y_prob),
        "Average Precision": average_precision_score(y_true, y_prob)
    }

baseline_fitted = {}
baseline_rows = []
baseline_times = {}

# Histogram-based Gradient Boosting expects dense input in common scikit-learn versions.
# Convert once here; this dataset is small enough for this feature matrix in Colab.
X_train_baseline = X_train_p.toarray() if hasattr(X_train_p, "toarray") else np.asarray(X_train_p)
X_val_baseline = X_val_p.toarray() if hasattr(X_val_p, "toarray") else np.asarray(X_val_p)

for name, model in baseline_models.items():
    start = time.time()
    model.fit(X_train_baseline, y_train)
    baseline_times[name] = time.time() - start
    baseline_fitted[name] = model
    pred = model.predict(X_val_baseline)
    prob = model.predict_proba(X_val_baseline)[:, 1]
    baseline_rows.append(calculate_metrics(name, y_val, pred, prob))

baseline_results = pd.DataFrame(baseline_rows).sort_values("ROC-AUC", ascending=False)
display(baseline_results.round(4))
print("Training times (seconds):", {k: round(v, 2) for k, v in baseline_times.items()})

In [ ]:
baseline_results.set_index("Model")[["Precision", "Recall", "F1", "ROC-AUC", "Average Precision"]].plot(
    kind="bar", figsize=(11, 5), ylim=(0, 1)
)
plt.title("Traditional model comparison — validation set")
plt.ylabel("Score")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "traditional_model_comparison.png", dpi=160)
plt.show()

## 10. Hyperparameter tuning with cross-validation

We tune Random Forest settings using 3-fold cross-validation and F1-score. F1 balances precision and recall. The search uses the raw training columns and a pipeline containing preprocessing, which reduces the risk of leakage inside cross-validation.

This is a compact grid to keep execution manageable. If you have time, add more values or tune a second algorithm and compare the results.

In [ ]:
rf_pipeline = Pipeline([
    ("preprocessor", make_preprocessor()),
    ("classifier", RandomForestClassifier(
        class_weight="balanced", random_state=SEED, n_jobs=-1
    ))
])

rf_param_grid = {
    "classifier__n_estimators": [100, 200],
    "classifier__max_depth": [None, 15],
    "classifier__min_samples_leaf": [1, 5]
}

rf_search = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=rf_param_grid,
    scoring="f1",
    cv=3,
    n_jobs=-1,
    verbose=1,
    return_train_score=True
)
start = time.time()
rf_search.fit(X_train, y_train)
print(f"Grid search completed in {time.time() - start:.1f} seconds")
print("Best parameters:", rf_search.best_params_)
print("Best mean cross-validation F1:", round(rf_search.best_score_, 4))

In [ ]:
cv_results = pd.DataFrame(rf_search.cv_results_)
display(cv_results[[
    "params", "mean_train_score", "mean_test_score", "std_test_score", "rank_test_score"
]].sort_values("rank_test_score").head(10))

## 11. Prepare TensorFlow `tf.data` datasets

The neural network uses the preprocessed features. One-hot encoding may produce a sparse matrix; TensorFlow's dense model input is easiest to use with dense `float32` arrays. This is generally manageable for this dataset in Colab.

We use batching, shuffling for training, and prefetching to prepare data efficiently.

In [ ]:
def dense_float32(matrix):
    if hasattr(matrix, "toarray"):
        matrix = matrix.toarray()
    return np.asarray(matrix, dtype=np.float32)

X_train_dl = dense_float32(X_train_p)
X_val_dl = dense_float32(X_val_p)
X_test_dl = dense_float32(X_test_p)

y_train_np = y_train.to_numpy(dtype=np.float32)
y_val_np = y_val.to_numpy(dtype=np.float32)
y_test_np = y_test.to_numpy(dtype=np.float32)

BATCH_SIZE = 128
train_ds = (
    tf.data.Dataset.from_tensor_slices((X_train_dl, y_train_np))
    .shuffle(buffer_size=min(len(X_train_dl), 10000), seed=SEED)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)
val_ds = (
    tf.data.Dataset.from_tensor_slices((X_val_dl, y_val_np))
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)
test_ds = (
    tf.data.Dataset.from_tensor_slices((X_test_dl, y_test_np))
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

print("Training batches:", tf.data.experimental.cardinality(train_ds).numpy())
print("Validation batches:", tf.data.experimental.cardinality(val_ds).numpy())

## 12. Build and train the Keras neural network

The model uses the Keras Sequential API:
- 128-neuron hidden layer with ReLU
- Dropout regularization
- 64-neuron hidden layer with ReLU
- One sigmoid output for the probability of subscription

Binary cross-entropy is the loss for binary classification. Early stopping monitors validation loss and restores the best weights.

In [ ]:
tf.keras.utils.set_random_seed(SEED)
dl_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train_dl.shape[1],)),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.30),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dropout(0.20),
    tf.keras.layers.Dense(1, activation="sigmoid")
])

dl_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name="accuracy"),
        tf.keras.metrics.AUC(name="auc"),
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall")
    ]
)
dl_model.summary()

In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=5, restore_best_weights=True
)
history = dl_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history["loss"], label="Training loss")
axes[0].plot(history.history["val_loss"], label="Validation loss")
axes[0].set_title("Neural network loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Binary cross-entropy")
axes[0].legend()

axes[1].plot(history.history["auc"], label="Training AUC")
axes[1].plot(history.history["val_auc"], label="Validation AUC")
axes[1].set_title("Neural network AUC")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("AUC")
axes[1].legend()
plt.tight_layout()
plt.savefig(FIGURE_DIR / "neural_network_learning_curves.png", dpi=160)
plt.show()

## 13. Compare selected models on validation data

The Random Forest selected by cross-validation and the neural network are compared on the validation set. A default threshold of 0.5 converts probabilities to labels. Threshold choice is a modeling decision: changing it can trade precision for recall. Do not use test results to choose the threshold.

In [ ]:
best_rf = rf_search.best_estimator_

rf_val_prob = best_rf.predict_proba(X_val)[:, 1]
rf_val_pred = (rf_val_prob >= 0.5).astype(int)
nn_val_prob = dl_model.predict(X_val_dl, verbose=0).ravel()
nn_val_pred = (nn_val_prob >= 0.5).astype(int)

validation_results = pd.DataFrame([
    calculate_metrics("Tuned Random Forest", y_val, rf_val_pred, rf_val_prob),
    calculate_metrics("Neural Network", y_val, nn_val_pred, nn_val_prob)
]).sort_values("ROC-AUC", ascending=False)

display(validation_results.round(4))
print("Neural network classification report:")
print(classification_report(
    y_val, nn_val_pred, target_names=["No subscription", "Subscription"], zero_division=0
))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
ConfusionMatrixDisplay.from_predictions(
    y_val, rf_val_pred, display_labels=["No", "Yes"], cmap="Blues", ax=axes[0]
)
axes[0].set_title("Tuned Random Forest — validation")
ConfusionMatrixDisplay.from_predictions(
    y_val, nn_val_pred, display_labels=["No", "Yes"], cmap="Greens", ax=axes[1]
)
axes[1].set_title("Neural Network — validation")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "validation_confusion_matrices.png", dpi=160)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
RocCurveDisplay.from_predictions(y_val, rf_val_prob, name="Tuned Random Forest", ax=ax)
RocCurveDisplay.from_predictions(y_val, nn_val_prob, name="Neural Network", ax=ax)
ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Chance")
ax.set_title("Validation ROC curves")
ax.legend()
plt.tight_layout()
plt.savefig(FIGURE_DIR / "validation_roc_curves.png", dpi=160)
plt.show()

fig, ax = plt.subplots(figsize=(7, 6))
for label, probs in [("Tuned Random Forest", rf_val_prob), ("Neural Network", nn_val_prob)]:
    precision, recall, _ = precision_recall_curve(y_val, probs)
    ax.plot(recall, precision, label=f"{label} (AP={average_precision_score(y_val, probs):.3f})")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Validation precision-recall curves")
ax.legend()
plt.tight_layout()
plt.savefig(FIGURE_DIR / "validation_precision_recall_curves.png", dpi=160)
plt.show()

## 14. Learning curve for the tuned Random Forest

A learning curve compares training and cross-validation performance at different training-set sizes. A large gap between training and validation performance can suggest overfitting; both scores being low can suggest underfitting. Interpret the curve alongside the dataset and model complexity.

This cell may take several minutes because it fits multiple models.

In [ ]:
sizes, train_scores, cv_scores = learning_curve(
    estimator=rf_pipeline,
    X=X_train,
    y=y_train,
    train_sizes=np.linspace(0.2, 1.0, 5),
    cv=3,
    scoring="f1",
    n_jobs=-1,
    shuffle=True,
    random_state=SEED
)

train_mean, train_std = train_scores.mean(axis=1), train_scores.std(axis=1)
cv_mean, cv_std = cv_scores.mean(axis=1), cv_scores.std(axis=1)

plt.figure(figsize=(8, 5))
plt.plot(sizes, train_mean, marker="o", label="Training F1")
plt.plot(sizes, cv_mean, marker="o", label="Cross-validation F1")
plt.fill_between(sizes, train_mean-train_std, train_mean+train_std, alpha=0.15)
plt.fill_between(sizes, cv_mean-cv_std, cv_mean+cv_std, alpha=0.15)
plt.xlabel("Number of training examples")
plt.ylabel("F1-score")
plt.title("Random Forest learning curve")
plt.legend()
plt.tight_layout()
plt.savefig(FIGURE_DIR / "random_forest_learning_curve.png", dpi=160)
plt.show()

## 15. Final test evaluation

After comparing models and selecting settings, evaluate the tuned Random Forest and neural network on the held-out test set. These are the final comparison results. Do not continue tuning based on test performance.

In [ ]:
rf_test_prob = best_rf.predict_proba(X_test)[:, 1]
rf_test_pred = (rf_test_prob >= 0.5).astype(int)
nn_test_prob = dl_model.predict(X_test_dl, verbose=0).ravel()
nn_test_pred = (nn_test_prob >= 0.5).astype(int)

final_results = pd.DataFrame([
    calculate_metrics("Tuned Random Forest", y_test, rf_test_pred, rf_test_prob),
    calculate_metrics("Neural Network", y_test, nn_test_pred, nn_test_prob)
]).sort_values("ROC-AUC", ascending=False)

display(final_results.round(4))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
RocCurveDisplay.from_predictions(y_test, rf_test_prob, name="Tuned Random Forest", ax=ax)
RocCurveDisplay.from_predictions(y_test, nn_test_prob, name="Neural Network", ax=ax)
ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Chance")
ax.set_title("Final test ROC curves")
ax.legend()
plt.tight_layout()
plt.savefig(FIGURE_DIR / "final_test_roc_curves.png", dpi=160)
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, rf_test_pred, display_labels=["No", "Yes"], cmap="Blues", ax=axes[0]
)
axes[0].set_title("Tuned Random Forest — test")
ConfusionMatrixDisplay.from_predictions(
    y_test, nn_test_pred, display_labels=["No", "Yes"], cmap="Greens", ax=axes[1]
)
axes[1].set_title("Neural Network — test")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "final_test_confusion_matrices.png", dpi=160)
plt.show()

## 16. Error analysis

A **false negative** is a customer who subscribed but the model predicted no. A **false positive** is a customer predicted to subscribe who did not. Inspect sample errors to understand where the model struggles. Do not infer causation from a few examples.

In [ ]:
error_table = X_test.copy()
error_table["actual"] = y_test
error_table["rf_probability"] = rf_test_prob
error_table["rf_prediction"] = rf_test_pred
error_table["nn_probability"] = nn_test_prob
error_table["nn_prediction"] = nn_test_pred
error_table["rf_error"] = error_table["actual"] != error_table["rf_prediction"]
error_table["nn_error"] = error_table["actual"] != error_table["nn_prediction"]

print("Random Forest errors:", int(error_table["rf_error"].sum()))
print("Neural network errors:", int(error_table["nn_error"].sum()))
print("Random Forest false negatives:", int(((y_test == 1) & (rf_test_pred == 0)).sum()))
print("Random Forest false positives:", int(((y_test == 0) & (rf_test_pred == 1)).sum()))
display(error_table[error_table["rf_error"]].head(20))

## 17. Save the models, metrics, and figures

The Scikit-learn pipeline includes preprocessing and the trained classifier, so it can accept raw feature rows. The Keras model is saved in `.keras` format. A copy of the fitted preprocessor is also saved for reference.

Run this cell after the training and evaluation sections have completed.

In [ ]:
joblib.dump(best_rf, ARTIFACT_DIR / "best_traditional_model.joblib")
joblib.dump(best_rf.named_steps["preprocessor"], ARTIFACT_DIR / "traditional_preprocessor.joblib")
dl_model.save(ARTIFACT_DIR / "best_deep_learning_model.keras")
final_results.to_csv(ARTIFACT_DIR / "final_test_results.csv", index=False)
validation_results.to_csv(ARTIFACT_DIR / "validation_results.csv", index=False)
baseline_results.to_csv(ARTIFACT_DIR / "traditional_baseline_results.csv", index=False)

print("Saved artifacts:")
for path in sorted(ARTIFACT_DIR.iterdir()):
    print(f"- {path} ({path.stat().st_size / 1024:.1f} KB)")
print("\nSaved figures:")
for path in sorted(FIGURE_DIR.iterdir()):
    print("-", path)

In [ ]:
# Verify saved models can be loaded and make predictions.
loaded_rf = joblib.load(ARTIFACT_DIR / "best_traditional_model.joblib")
loaded_nn = tf.keras.models.load_model(ARTIFACT_DIR / "best_deep_learning_model.keras")

print("Traditional model sample probabilities:", loaded_rf.predict_proba(X_test.head(3))[:, 1])
print("Neural network sample probabilities:", loaded_nn.predict(X_test_dl[:3], verbose=0).ravel())

In [ ]:
# Create one ZIP containing model artifacts and figures for convenient transfer to Google Drive.
import zipfile
bundle_path = Path("bank_marketing_project_outputs.zip")
with zipfile.ZipFile(bundle_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for folder in [ARTIFACT_DIR, FIGURE_DIR]:
        for path in folder.rglob("*"):
            if path.is_file():
                zf.write(path, arcname=str(path))
print("Created:", bundle_path.resolve())

# In Google Colab, uncomment these two lines to download the ZIP to your computer:
# from google.colab import files
# files.download(str(bundle_path))


## 18. Report-writing worksheet: interpret your actual outputs

Complete these notes using the values and figures generated by your own run. Do not invent or copy scores from another person's run.

1. **Problem statement:** Why is predicting term-deposit subscription a classification task? Who could use such a model, and what decisions might it support?
2. **Dataset:** What is the source, raw row count, feature count, target distribution, and dataset limitation?
3. **Preprocessing:** Why use imputation, scaling, and one-hot encoding? Why exclude `duration`? How was data leakage avoided?
4. **Traditional ML:** Compare Logistic Regression, Random Forest, and Gradient Boosting. Which metrics matter most for this imbalanced target?
5. **Tuning:** What were the best Random Forest settings? Compare the cross-validation score with validation performance.
6. **Deep learning:** Describe the network architecture, optimizer, loss, batch size, early stopping, and learning curves.
7. **Results:** Compare the selected models on the untouched test set. Discuss precision, recall, F1, ROC-AUC, and average precision.
8. **Error analysis:** What do the confusion matrices show? Explain false positives and false negatives in the project's context.
9. **Limitations:** The dataset is historical and reflects particular banking campaigns and a particular setting. Consider temporal drift, class imbalance, unknown categories, and generalization to current customers.
10. **Future work:** Suggest a further justified experiment, such as threshold tuning, a broader hyperparameter search, calibration, or time-based validation.

## References and documentation

- UCI Machine Learning Repository — Bank Marketing: https://archive.ics.uci.edu/dataset/222/bank+marketing
- Scikit-learn user guide: https://scikit-learn.org/stable/
- TensorFlow `tf.data` guide: https://www.tensorflow.org/guide/data
- Keras Sequential API: https://keras.io/guides/sequential_model/

## Submission checklist

- [ ] Run the full notebook and save a copy with outputs.
- [ ] Add at least one further experiment and interpret it.
- [ ] Create a public/shared GitHub repository with this notebook, README, and `requirements.txt`.
- [ ] Do **not** commit the raw dataset to GitHub.
- [ ] Upload the zipped raw dataset and both trained model artifacts to a Google Drive folder set to “Anyone with the link can view.”
- [ ] Include working GitHub, Drive, and video links on the first page/header of the IEEE-style report.
- [ ] Write a 3,500–5,000 word report based on your own analysis.
- [ ] Record a 5–10 minute presentation video.
- [ ] Follow your course's AI policy and make sure the report reflects your own analysis and writing.